# Polymarket Insider-Signal Audit (2025-2026)

This notebook audits completed Polymarket events for pre-resolution price patterns that may indicate potential information-leak signals.

It explicitly includes event clusters for:
- Maduro-related contracts
- Iran strike-timing contracts
- Khamenei/succession contracts


## Scope

1. Pull closed markets in 2025-2026 from Gamma API.
2. Build target event set from seeded IDs + keyword matches.
3. Pull CLOB minute-level price history for each event's `Yes` token.
4. Compute anomaly metrics in late windows (`T-24h`, `T-6h`, `T-1h`) vs baseline (`T-30d` to `T-7d`, with fallback).
5. Export ranked results to JSON/CSV in `reports/`.

`T` is the market close timestamp (`closedTime` when present, otherwise `endDate`).


In [1]:
import csv
import json
import math
import os
import re
import statistics
import subprocess
import time
import urllib.error
import urllib.parse
import urllib.request
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

GAMMA_API = "https://gamma-api.polymarket.com"
CLOB_API = "https://clob.polymarket.com"
USER_AGENT = "Mozilla/5.0 (compatible; NewsroomNotebook/1.0)"

START_DATE_MIN = "2025-01-01T00:00:00Z"
START_DATE_MAX = "2026-12-31T23:59:59Z"

PAGE_SIZE = 200
MAX_PAGES = 120

# Explicitly seed the three case clusters and close variants around them.
SEED_MARKET_IDS = [
    # Iran strike timing cluster
    "532742",  # Israel military action against Iran before July?
    "532741",  # US military action against Iran before July?
    "1465967", # Will US or Israel strike Iran on March 2, 2026?
    "1465968", # Will US or Israel strike Iran on March 3, 2026?
    # Khamenei/succession cluster
    "1469316", # Will Iran name a successor to Khamenei by March 2?
    "1493097", # Will Iran announce a new supreme leader on March 3, 2026?
    "527803",  # Will Ali Khamenei be the first leader out in 2025?
    # Maduro cluster
    "688265",  # Maduro in U.S. custody by December 31?
    "687779",  # U.S. operation to capture Maduro in 2025?
    "1096292"  # Nicolas Maduro released from custody by Jan 9, 2026?
]

KEYWORD_CASE_GROUPS = {
    "maduro": ["maduro", "venezuela"],
    "iran_strikes": ["iran", "strike", "military action", "us or israel"],
    "khamenei": ["khamenei", "supreme leader", "successor"],
}

CACHE_DIR = "data"
REPORT_DIR = "reports"

S3_BUCKET = "asoba-llm-cache"
S3_LATEST_PREFIX = "zorora/demos/latest"
S3_KEYS = {
    "closed_markets": "data/closed_markets_2025_2026.json",
    "target_markets": "data/polymarket_target_markets_2025_2026.json",
    "price_histories": "data/polymarket_price_histories_2025_2026.json",
    "report_json": "reports/polymarket_signal_audit_2025_2026.json",
    "report_csv": "reports/polymarket_signal_audit_2025_2026.csv",
}

# When True, notebook attempts to hydrate local cache from S3 before fetching APIs.
DOWNLOAD_CACHE_FROM_S3 = True
# When True, analysis cell loads downloaded latest report instead of re-querying APIs.
USE_PRECOMPUTED_REPORT_IF_AVAILABLE = True
HF_NEHANDA_ENDPOINT_URL = "https://tmlbj0s0e5y41gft.us-east-1.aws.endpoints.huggingface.cloud"
HF_NEHANDA_MODEL_ID = "asoba/nehanda-v1-7b"
HF_TOKEN_ENV_KEYS = ["HF_TOKEN", "HUGGINGFACE_API_KEY", "HUGGINGFACEHUB_API_TOKEN"]
# Endpoint generation should be stochastic; only fallback path is deterministic.
HF_ANALYTIC_REPORT_MAX_TOKENS = 1400
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)


In [2]:
def api_get_json(url: str, params: Optional[Dict[str, Any]] = None, headers: Optional[Dict[str, str]] = None, timeout: int = 30) -> Any:
    request_headers = {"User-Agent": USER_AGENT}
    if headers:
        request_headers.update(headers)

    query = urllib.parse.urlencode(params or {}, doseq=True)
    full_url = f"{url}?{query}" if query else url
    request = urllib.request.Request(full_url, headers=request_headers)

    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            payload = response.read().decode("utf-8", "replace")
    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8", "replace")
        raise RuntimeError(f"HTTP {exc.code} for {full_url}: {body[:300]}") from exc

    return json.loads(payload, strict=False)


def parse_maybe_json(value: Any) -> Any:
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return value
    return value


def parse_ts(ts_value: Optional[str]) -> Optional[int]:
    if not ts_value:
        return None
    ts = ts_value.strip()
    if " " in ts and "+" in ts and "T" not in ts:
        ts = ts.replace(" ", "T")
    if ts.endswith("Z"):
        ts = ts.replace("Z", "+00:00")
    return int(datetime.fromisoformat(ts).timestamp())


def choose_event_ts(market: Dict[str, Any]) -> Optional[int]:
    return parse_ts(market.get("closedTime")) or parse_ts(market.get("endDate"))


def choose_yes_token(market: Dict[str, Any]) -> Optional[str]:
    outcomes = parse_maybe_json(market.get("outcomes", [])) or []
    token_ids = parse_maybe_json(market.get("clobTokenIds", [])) or []
    if not token_ids:
        return None
    if isinstance(outcomes, list) and outcomes:
        lowered = [str(x).lower() for x in outcomes]
        if "yes" in lowered:
            idx = lowered.index("yes")
            if idx < len(token_ids):
                return str(token_ids[idx])
    return str(token_ids[0])


def fetch_market_by_id(market_id: str) -> Optional[Dict[str, Any]]:
    payload = api_get_json(
        f"{GAMMA_API}/markets",
        params={"id": market_id, "limit": 1, "offset": 0},
    )
    if isinstance(payload, list) and payload:
        return payload[0]
    return None


In [3]:
def s3_uri_for_key(relative_key: str) -> str:
    return f"s3://{S3_BUCKET}/{S3_LATEST_PREFIX}/{relative_key}"


def run_command(cmd: List[str]) -> Tuple[int, str, str]:
    completed = subprocess.run(cmd, capture_output=True, text=True)
    return completed.returncode, completed.stdout.strip(), completed.stderr.strip()


def download_latest_bundle_from_s3(
    include_closed_markets: bool = True,
    include_target_markets: bool = True,
    include_price_histories: bool = True,
    include_reports: bool = True,
    overwrite: bool = False,
) -> List[str]:
    artifacts: List[Tuple[str, str]] = []
    if include_closed_markets:
        artifacts.append((S3_KEYS["closed_markets"], os.path.join(CACHE_DIR, "closed_markets_2025_2026.json")))
    if include_target_markets:
        artifacts.append((S3_KEYS["target_markets"], os.path.join(CACHE_DIR, "polymarket_target_markets_2025_2026_latest.json")))
    if include_price_histories:
        artifacts.append((S3_KEYS["price_histories"], os.path.join(CACHE_DIR, "polymarket_price_histories_2025_2026_latest.json")))
    if include_reports:
        artifacts.append((S3_KEYS["report_json"], os.path.join(REPORT_DIR, "polymarket_signal_audit_2025_2026_latest.json")))
        artifacts.append((S3_KEYS["report_csv"], os.path.join(REPORT_DIR, "polymarket_signal_audit_2025_2026_latest.csv")))

    downloaded: List[str] = []
    for relative_key, local_path in artifacts:
        if not overwrite and os.path.exists(local_path):
            continue
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        s3_uri = s3_uri_for_key(relative_key)
        rc, _, stderr = run_command(["aws", "s3", "cp", s3_uri, local_path])
        if rc == 0:
            downloaded.append(local_path)
            print(f"Downloaded: {s3_uri} -> {local_path}")
        else:
            print(f"S3 download skipped/failed for {s3_uri}: {stderr[:200]}")
    return downloaded


def fetch_closed_markets_2025_2026(max_pages: int = MAX_PAGES, page_size: int = PAGE_SIZE) -> List[Dict[str, Any]]:
    markets: List[Dict[str, Any]] = []
    for page in range(max_pages):
        offset = page * page_size
        params = {
            "limit": page_size,
            "offset": offset,
            "closed": "true",
            "start_date_min": START_DATE_MIN,
            "start_date_max": START_DATE_MAX,
        }
        batch = api_get_json(f"{GAMMA_API}/markets", params=params)
        if not isinstance(batch, list) or not batch:
            break
        markets.extend(batch)
        print(f"Fetched page={page} offset={offset} batch={len(batch)} cumulative={len(markets)}")
        if len(batch) < page_size:
            break
    return markets


def load_or_fetch_closed_markets(refresh: bool = False) -> List[Dict[str, Any]]:
    cache_path = os.path.join(CACHE_DIR, "closed_markets_2025_2026.json")

    if not refresh and not os.path.exists(cache_path) and DOWNLOAD_CACHE_FROM_S3:
        download_latest_bundle_from_s3(
            include_closed_markets=True,
            include_target_markets=False,
            include_price_histories=False,
            include_reports=False,
            overwrite=False,
        )

    if not refresh and os.path.exists(cache_path):
        with open(cache_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(f"Loaded {len(data)} markets from cache: {cache_path}")
        return data

    data = fetch_closed_markets_2025_2026()
    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(data, f)
    print(f"Cached {len(data)} markets to: {cache_path}")
    return data


def text_for_market(market: Dict[str, Any]) -> str:
    return " ".join([
        str(market.get("question", "")),
        str(market.get("description", "")),
        str(market.get("slug", "")),
    ]).lower()


def build_target_markets(
    all_markets: List[Dict[str, Any]],
    seed_market_ids: List[str],
    keyword_case_groups: Dict[str, List[str]],
) -> Tuple[List[Dict[str, Any]], List[str]]:
    by_id = {str(m.get("id")): m for m in all_markets if m.get("id") is not None}
    selected: Dict[str, Dict[str, Any]] = {}
    missing_seed_ids: List[str] = []

    for market_id in seed_market_ids:
        if market_id in by_id:
            selected[market_id] = by_id[market_id]
        else:
            fetched = fetch_market_by_id(market_id)
            if fetched:
                selected[market_id] = fetched
            else:
                missing_seed_ids.append(market_id)

    for case_name, keywords in keyword_case_groups.items():
        regexes = [re.compile(re.escape(k.lower())) for k in keywords]
        for market in all_markets:
            market_id = str(market.get("id"))
            text = text_for_market(market)
            if all(r.search(text) for r in regexes[:1]):
                if any(r.search(text) for r in regexes):
                    selected[market_id] = market

    target_markets = list(selected.values())
    target_markets.sort(key=lambda m: choose_event_ts(m) or 0)
    return target_markets, missing_seed_ids


if DOWNLOAD_CACHE_FROM_S3:
    downloaded_artifacts = download_latest_bundle_from_s3(
        include_closed_markets=not os.path.exists(os.path.join(CACHE_DIR, "closed_markets_2025_2026.json")),
        include_target_markets=not os.path.exists(os.path.join(CACHE_DIR, "polymarket_target_markets_2025_2026_latest.json")),
        include_price_histories=not os.path.exists(os.path.join(CACHE_DIR, "polymarket_price_histories_2025_2026_latest.json")),
        include_reports=not os.path.exists(os.path.join(REPORT_DIR, "polymarket_signal_audit_2025_2026_latest.json")),
        overwrite=False,
    )
    if downloaded_artifacts:
        print(f"Downloaded {len(downloaded_artifacts)} S3 cache artifacts.")

all_closed_markets = load_or_fetch_closed_markets(refresh=False)
target_markets, missing_seed_ids = build_target_markets(all_closed_markets, SEED_MARKET_IDS, KEYWORD_CASE_GROUPS)

print(f"Total closed markets loaded: {len(all_closed_markets)}")
print(f"Target markets selected: {len(target_markets)}")
if missing_seed_ids:
    print(f"Missing seed ids: {missing_seed_ids}")

for market in target_markets[:40]:
    print(market.get("id"), market.get("closedTime"), market.get("question"))


Loaded 24000 markets from cache: data/closed_markets_2025_2026.json


Total closed markets loaded: 24000
Target markets selected: 35
521790 2025-02-05 02:22:23+00 Will Trump say "Iran" during his February 4 press conference with Netanyahu?
524357 2025-02-25 22:35:40+00 Will Karoline Leavitt say "Iran" during next White House press briefing?
528759 2025-03-22 06:28:59+00 Israeli military action against Iran by Friday?
528764 2025-03-22 06:29:09+00 Fact Check: Did the U.S. hit Iranian spy ship?
531640 2025-03-28 03:45:10+00 Will Trump say "Iran" during today's Iftar Dinner?
531897 2025-03-29 23:25:40+00 Emmers vs. Miranda
532762 2025-04-01 01:10:15+00 Will Trump say "Iran" during Executive Order signing today?
528763 2025-04-01 07:31:06+00 US military action against Iran before April?
528760 2025-04-01 07:56:14+00 Israel military action against Iranian nuclear facility in March?
528761 2025-04-01 09:22:12+00 Israel military action against Iran in March?
534857 2025-04-07 21:54:17+00 Will Trump say "Iran" 3+ times during Netanyahu events today?
535819 2025-

In [4]:
def fetch_price_history(token_id: str, start_ts: int, end_ts: int, fidelity_candidates: Optional[List[int]] = None) -> Tuple[List[Dict[str, Any]], Optional[int], Optional[str]]:
    if fidelity_candidates is None:
        fidelity_candidates = [1, 5, 15, 60, 240, 1440]

    last_error: Optional[str] = None
    for fidelity in fidelity_candidates:
        payload = api_get_json(
            f"{CLOB_API}/prices-history",
            params={
                "market": token_id,
                "startTs": start_ts,
                "endTs": end_ts,
                "fidelity": fidelity,
            },
        )
        if isinstance(payload, dict) and "history" in payload:
            return payload["history"], fidelity, None
        if isinstance(payload, dict) and "error" in payload:
            last_error = str(payload["error"])
            if "interval is too long" in last_error.lower():
                continue
            return [], fidelity, last_error
        return [], fidelity, "Unexpected prices-history payload"

    return [], None, last_error or "No valid fidelity found"


def compute_signal_metrics(points: List[Dict[str, Any]], event_ts: int) -> Dict[str, Any]:
    if not points:
        return {"has_history": False}

    series = sorted([(int(p["t"]), float(p["p"])) for p in points if "t" in p and "p" in p], key=lambda x: x[0])
    if not series:
        return {"has_history": False}

    day = 24 * 3600
    hour = 3600

    baseline = [p for t, p in series if (event_ts - 30 * day) <= t < (event_ts - 7 * day)]
    if len(baseline) < 25:
        baseline = [p for t, p in series if (event_ts - 14 * day) <= t < (event_ts - 2 * day)]

    pre24h = [(t, p) for t, p in series if (event_ts - 24 * hour) <= t <= event_ts]
    pre6h = [(t, p) for t, p in series if (event_ts - 6 * hour) <= t <= event_ts]
    pre1h = [(t, p) for t, p in series if (event_ts - 1 * hour) <= t <= event_ts]

    def last_before(ts_cutoff: int) -> Optional[float]:
        candidates = [p for t, p in series if t <= ts_cutoff]
        if not candidates:
            return None
        return candidates[-1]

    baseline_mean = statistics.mean(baseline) if baseline else None
    baseline_sd = statistics.pstdev(baseline) if len(baseline) > 1 else None

    p_24h = last_before(event_ts - 24 * hour)
    p_6h = last_before(event_ts - 6 * hour)
    p_1h = last_before(event_ts - 1 * hour)
    p_last = last_before(event_ts)

    max_6h = max((p for _, p in pre6h), default=None)
    min_6h = min((p for _, p in pre6h), default=None)

    if baseline_mean is not None and baseline_sd not in (None, 0) and max_6h is not None:
        max_6h_z = (max_6h - baseline_mean) / baseline_sd
    else:
        max_6h_z = None

    def hours_before_close_cross(threshold: float) -> Optional[float]:
        cross_ts = None
        for t, p in series:
            if p >= threshold:
                cross_ts = t
                break
        if cross_ts is None:
            return None
        return round((event_ts - cross_ts) / 3600.0, 2)

    return {
        "has_history": True,
        "n_points": len(series),
        "baseline_mean": baseline_mean,
        "baseline_sd": baseline_sd,
        "p_24h": p_24h,
        "p_6h": p_6h,
        "p_1h": p_1h,
        "p_last": p_last,
        "pre24h_mean": statistics.mean([p for _, p in pre24h]) if pre24h else None,
        "pre6h_mean": statistics.mean([p for _, p in pre6h]) if pre6h else None,
        "pre1h_mean": statistics.mean([p for _, p in pre1h]) if pre1h else None,
        "max_6h": max_6h,
        "min_6h": min_6h,
        "range_6h": (max_6h - min_6h) if (max_6h is not None and min_6h is not None) else None,
        "max_6h_z": max_6h_z,
        "jump_24h_to_1h": (p_1h - p_24h) if (p_1h is not None and p_24h is not None) else None,
        "jump_6h_to_1h": (p_1h - p_6h) if (p_1h is not None and p_6h is not None) else None,
        "hours_before_close_cross_1pct": hours_before_close_cross(0.01),
        "hours_before_close_cross_2pct": hours_before_close_cross(0.02),
        "hours_before_close_cross_5pct": hours_before_close_cross(0.05),
        "hours_before_close_cross_10pct": hours_before_close_cross(0.10),
    }


def safe_num(value: Optional[float], default: float = 0.0) -> float:
    if value is None:
        return default
    if isinstance(value, float) and math.isnan(value):
        return default
    return float(value)


def signal_score(row: Dict[str, Any]) -> float:
    z_component = max(0.0, safe_num(row.get("max_6h_z"), 0.0))
    jump_component = max(0.0, safe_num(row.get("jump_24h_to_1h"), 0.0) * 100.0)
    range_component = max(0.0, safe_num(row.get("range_6h"), 0.0) * 100.0)
    return round((0.50 * z_component) + (0.30 * jump_component) + (0.20 * range_component), 4)


In [5]:
def audit_market(market: Dict[str, Any]) -> Dict[str, Any]:
    market_id = str(market.get("id"))
    question = str(market.get("question", ""))
    slug = str(market.get("slug", ""))
    closed_time = market.get("closedTime")
    end_date = market.get("endDate")
    start_date = market.get("startDate")
    volume_num = market.get("volumeNum") if market.get("volumeNum") is not None else market.get("volume")

    event_ts = choose_event_ts(market)
    yes_token = choose_yes_token(market)

    row: Dict[str, Any] = {
        "market_id": market_id,
        "question": question,
        "slug": slug,
        "start_date": start_date,
        "end_date": end_date,
        "closed_time": closed_time,
        "event_ts": event_ts,
        "volume_num": volume_num,
        "yes_token": yes_token,
    }

    if event_ts is None or yes_token is None:
        row["error"] = "Missing event timestamp or yes token"
        return row

    history_start_ts = event_ts - (30 * 24 * 3600)
    history, fidelity_used, history_error = fetch_price_history(yes_token, history_start_ts, event_ts)
    row["history_fidelity"] = fidelity_used
    row["history_error"] = history_error

    metrics = compute_signal_metrics(history, event_ts)
    row.update(metrics)
    row["signal_score"] = signal_score(row)
    return row


precomputed_report_path = os.path.join(REPORT_DIR, "polymarket_signal_audit_2025_2026_latest.json")
audit_rows: List[Dict[str, Any]] = []

if USE_PRECOMPUTED_REPORT_IF_AVAILABLE and os.path.exists(precomputed_report_path):
    with open(precomputed_report_path, "r", encoding="utf-8") as f:
        audit_rows = json.load(f)
    print(f"Loaded precomputed audit rows from: {precomputed_report_path} ({len(audit_rows)} rows)")
else:
    for idx, market in enumerate(target_markets, start=1):
        try:
            row = audit_market(market)
            audit_rows.append(row)
        except Exception as exc:
            audit_rows.append({
                "market_id": str(market.get("id")),
                "question": str(market.get("question", "")),
                "error": str(exc),
            })
        if idx % 10 == 0:
            print(f"Audited {idx}/{len(target_markets)} markets")

audit_rows = sorted(audit_rows, key=lambda r: safe_num(r.get("signal_score"), 0.0), reverse=True)

try:
    import pandas as pd
except ImportError:
    pd = None

if pd is not None:
    df = pd.DataFrame(audit_rows)
    display_columns = [
        "signal_score", "market_id", "question", "closed_time",
        "max_6h_z", "jump_24h_to_1h", "range_6h",
        "p_24h", "p_1h", "p_last", "volume_num", "history_error"
    ]
    available_columns = [c for c in display_columns if c in df.columns]
    missing_columns = [c for c in display_columns if c not in df.columns]
    if missing_columns:
        print(f"Missing display columns in precomputed rows: {missing_columns}")
    display(df[available_columns].head(50))
else:
    for row in audit_rows[:50]:
        print(
            row.get("signal_score"),
            row.get("market_id"),
            row.get("closed_time"),
            row.get("max_6h_z"),
            row.get("question"),
        )



Loaded precomputed audit rows from: reports/polymarket_signal_audit_2025_2026_latest.json (35 rows)


Missing display columns in precomputed rows: ['history_error']


,signal_score,market_id,question,closed_time,max_6h_z,jump_24h_to_1h,range_6h,p_24h,p_1h,p_last,volume_num
0,41.1911,532742,Israel military action against Iran before July?,2025-06-13 03:24:45+00,28.132163,0.5745,0.4945,0.4250,0.9995,0.9995,4.506859e+06
1,31.9287,532741,US military action against Iran before July?,2025-06-22 03:03:06+00,10.437418,0.5710,0.4790,0.4275,0.9985,0.9995,2.991702e+07
2,27.1250,531897,Emmers vs. Miranda,2025-03-29 23:25:40+00,NaN,0.5245,0.5695,0.4750,0.9995,0.9995,3.255931e+04
3,16.8900,532762,"Will Trump say ""Iran"" during Executive Order s...",2025-04-01 01:10:15+00,NaN,NaN,0.8445,NaN,0.0005,0.0005,5.886788e+02
4,14.7900,534857,"Will Trump say ""Iran"" 3+ times during Netanyah...",2025-04-07 21:54:17+00,NaN,NaN,0.7395,NaN,0.9995,0.9995,7.327908e+03
5,12.3900,535819,"Will Trump say ""Iran"" 3+ times during his cabi...",2025-04-10 21:04:21+00,NaN,NaN,0.6195,NaN,0.0005,0.0005,4.974196e+03
6,10.3900,535981,"Will Karoline Leavitt say ""Iran"" 3+ times duri...",2025-04-11 20:26:21+00,NaN,NaN,0.5195,NaN,0.9950,0.9995,1.509833e+02
7,9.9900,521790,"Will Trump say ""Iran"" during his February 4 pr...",2025-02-05 02:22:23+00,NaN,NaN,0.4995,NaN,0.9995,0.9995,3.302589e+03
8,9.7900,531640,"Will Trump say ""Iran"" during today's Iftar Din...",2025-03-28 03:45:10+00,NaN,NaN,0.4895,NaN,0.0150,0.0005,2.412000e+03
9,8.0900,524357,"Will Karoline Leavitt say ""Iran"" during next W...",2025-02-25 22:35:40+00,-0.269963,-0.3445,0.4045,0.3450,0.0005,0.0005,1.451000e+03


In [6]:
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
json_path = os.path.join(REPORT_DIR, f"polymarket_signal_audit_2025_2026_{timestamp}.json")
csv_path = os.path.join(REPORT_DIR, f"polymarket_signal_audit_2025_2026_{timestamp}.csv")

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(audit_rows, f, indent=2)

fieldnames = sorted({k for row in audit_rows for k in row.keys()})
with open(csv_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(audit_rows)

print(f"Wrote JSON: {json_path}")
print(f"Wrote CSV:  {csv_path}")


Wrote JSON: reports/polymarket_signal_audit_2025_2026_20260307T010333Z.json
Wrote CSV:  reports/polymarket_signal_audit_2025_2026_20260307T010333Z.csv


## LLM Analytic Report (Nehanda Endpoint + Deterministic Fallback)\n\nThis cell attempts to generate a narrative report from `audit_rows` using the configured Hugging Face endpoint.\nIf endpoint/auth/parsing fails, it emits a deterministic template-based fallback report.\n

In [7]:
def get_hf_token() -> Optional[str]:
    for key in HF_TOKEN_ENV_KEYS:
        value = os.environ.get(key)
        if value:
            return value

    token_paths = [
        os.path.expanduser("~/.cache/huggingface/token"),
        "/Users/shingi/Workbench/zorora/config.py",
        "/Users/shingi/Workbench/zorora/.env",
        "/Users/shingi/Workbench/nehanda/.env",
    ]

    token_pattern = re.compile(
        r"(?:HF_TOKEN|HUGGINGFACE_API_KEY|HUGGINGFACEHUB_API_TOKEN)\s*[:=]\s*[\"']?([A-Za-z0-9_\-]+)",
        re.IGNORECASE,
    )

    for path in token_paths:
        if not os.path.exists(path):
            continue
        try:
            with open(path, "r", encoding="utf-8") as f:
                content = f.read()
        except Exception:
            continue

        if path.endswith("/token"):
            for line in content.splitlines():
                token = line.strip()
                if token:
                    return token
            continue

        match = token_pattern.search(content)
        if match:
            return match.group(1)

    return None


def post_json(url: str, payload: Dict[str, Any], headers: Dict[str, str], timeout: int = 120) -> Tuple[Optional[Any], Optional[str], Optional[int]]:
    body = json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(url, data=body, headers=headers, method="POST")
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            raw = response.read().decode("utf-8", "replace")
            try:
                return json.loads(raw, strict=False), None, response.status
            except json.JSONDecodeError:
                return raw, None, response.status
    except urllib.error.HTTPError as exc:
        err_body = exc.read().decode("utf-8", "replace")
        return None, f"HTTP {exc.code}: {err_body[:500]}", exc.code
    except Exception as exc:
        return None, str(exc), None


def extract_generated_text(payload: Any) -> Optional[str]:
    if payload is None:
        return None

    if isinstance(payload, str):
        return payload.strip() or None

    if isinstance(payload, list) and payload:
        first = payload[0]
        if isinstance(first, dict):
            for key in ("generated_text", "text", "summary_text"):
                value = first.get(key)
                if isinstance(value, str) and value.strip():
                    return value.strip()
        return None

    if isinstance(payload, dict):
        choices = payload.get("choices")
        if isinstance(choices, list) and choices:
            first_choice = choices[0]
            if isinstance(first_choice, dict):
                message = first_choice.get("message")
                if isinstance(message, dict):
                    content = message.get("content")
                    if isinstance(content, str) and content.strip():
                        return content.strip()
                text = first_choice.get("text")
                if isinstance(text, str) and text.strip():
                    return text.strip()

        for key in ("generated_text", "text", "summary_text"):
            value = payload.get(key)
            if isinstance(value, str) and value.strip():
                return value.strip()

        nested = payload.get("data")
        if nested is not None:
            return extract_generated_text(nested)

    return None


def infer_cluster(row: Dict[str, Any]) -> str:
    text = f"{row.get('question', '')} {row.get('slug', '')}".lower()
    if any(k in text for k in ["maduro", "venezuela"]):
        return "Maduro"
    if any(k in text for k in ["khamenei", "supreme leader", "successor"]):
        return "Khamenei"
    if any(k in text for k in ["iran", "military action", "strike"]):
        return "Iran Strikes"
    return "Other"


def is_speech_or_process_market(row: Dict[str, Any]) -> bool:
    text = f"{row.get('question', '')} {row.get('slug', '')}".lower()
    patterns = [
        r"\bwill .* say\b",
        r"press brief",
        r"cabinet meeting",
        r"executive order",
        r"iftar",
        r"fact check",
    ]
    return any(re.search(p, text) for p in patterns)


def is_off_topic_for_leak_hypothesis(row: Dict[str, Any]) -> bool:
    return infer_cluster(row) not in {"Maduro", "Khamenei", "Iran Strikes"}


def split_signal_buckets(rows: List[Dict[str, Any]]) -> Dict[str, List[Dict[str, Any]]]:
    ranked = sorted(rows, key=lambda r: safe_num(r.get("signal_score"), 0.0), reverse=True)

    high_confidence: List[Dict[str, Any]] = []
    likely_noise: List[Dict[str, Any]] = []
    supporting: List[Dict[str, Any]] = []

    for row in ranked:
        score = safe_num(row.get("signal_score"), 0.0)
        speech = is_speech_or_process_market(row)
        off_topic = is_off_topic_for_leak_hypothesis(row)
        z = row.get("max_6h_z")
        jump = row.get("jump_24h_to_1h")

        if speech or off_topic:
            likely_noise.append(row)
            continue

        strong_signal = (
            score >= 10
            and z is not None
            and jump is not None
            and safe_num(jump, 0.0) > 0
        )

        if strong_signal:
            high_confidence.append(row)
        elif score >= 1:
            supporting.append(row)
        else:
            likely_noise.append(row)

    return {
        "ranked": ranked,
        "high_confidence": high_confidence,
        "supporting": supporting,
        "likely_noise": likely_noise,
    }


def fmt_num(value: Any, digits: int = 4) -> str:
    if value is None:
        return "n/a"
    try:
        return f"{float(value):.{digits}f}"
    except Exception:
        return str(value)


def row_line(row: Dict[str, Any]) -> str:
    return (
        f"- `{row.get('market_id')}` score={fmt_num(row.get('signal_score'))}, "
        f"z={fmt_num(row.get('max_6h_z'), 3)}, jump24to1={fmt_num(row.get('jump_24h_to_1h'), 4)} :: "
        f"{row.get('question')}"
    )


def top_rows_for_cluster(rows: List[Dict[str, Any]], cluster: str, limit: int = 3) -> List[Dict[str, Any]]:
    selected = [r for r in rows if infer_cluster(r) == cluster]
    return sorted(selected, key=lambda r: safe_num(r.get("signal_score"), 0.0), reverse=True)[:limit]


def infer_hypothesis_verdict(rows: List[Dict[str, Any]], buckets: Optional[Dict[str, List[Dict[str, Any]]]] = None) -> str:
    buckets = buckets or split_signal_buckets(rows)
    ranked = buckets["ranked"]

    iran_top = top_rows_for_cluster(ranked, "Iran Strikes", limit=1)
    maduro_top = top_rows_for_cluster(ranked, "Maduro", limit=1)
    khamenei_top = top_rows_for_cluster(ranked, "Khamenei", limit=1)

    iran_score = safe_num(iran_top[0].get("signal_score"), 0.0) if iran_top else 0.0
    maduro_score = safe_num(maduro_top[0].get("signal_score"), 0.0) if maduro_top else 0.0
    khamenei_score = safe_num(khamenei_top[0].get("signal_score"), 0.0) if khamenei_top else 0.0

    if iran_score >= 10 and maduro_score < 10 and khamenei_score < 10:
        return (
            "Hypothesis verdict: partial support. The strongest anomaly signal appears in Iran strike-timing markets, "
            "while Maduro and Khamenei clusters do not show comparable strength in this run."
        )
    if iran_score >= 10 and (maduro_score >= 10 or khamenei_score >= 10):
        return (
            "Hypothesis verdict: broader support. Multiple clusters show strong pre-resolution anomalies, "
            "consistent with the leak-detector hypothesis across cases."
        )
    return (
        "Hypothesis verdict: not supported in this run. The score pattern does not show strong, repeated anomalies "
        "across the targeted case clusters."
    )


def summarize_scores(rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    scores = [safe_num(r.get("signal_score"), 0.0) for r in rows]
    if not scores:
        return {
            "n": 0,
            "mean": 0.0,
            "median": 0.0,
            "max": 0.0,
            "share_ge_1": 0.0,
            "share_ge_5": 0.0,
            "share_ge_10": 0.0,
            "share_ge_20": 0.0,
        }

    n = len(scores)
    return {
        "n": n,
        "mean": sum(scores) / n,
        "median": statistics.median(scores),
        "max": max(scores),
        "share_ge_1": sum(v >= 1 for v in scores) / n,
        "share_ge_5": sum(v >= 5 for v in scores) / n,
        "share_ge_10": sum(v >= 10 for v in scores) / n,
        "share_ge_20": sum(v >= 20 for v in scores) / n,
    }


def build_experiment_packet(rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    seeded_ids = {str(x) for x in SEED_MARKET_IDS}
    ranked = sorted(rows, key=lambda r: safe_num(r.get("signal_score"), 0.0), reverse=True)

    seeded = [r for r in ranked if str(r.get("market_id")) in seeded_ids]
    non_seeded = [r for r in ranked if str(r.get("market_id")) not in seeded_ids]
    non_seeded_event_eligible = [r for r in non_seeded if not is_speech_or_process_market(r)]

    by_cluster: Dict[str, List[Dict[str, Any]]] = {"Iran Strikes": [], "Maduro": [], "Khamenei": [], "Other": []}
    for row in ranked:
        by_cluster[infer_cluster(row)].append(row)

    cluster_top = {
        cluster: [
            {
                "market_id": r.get("market_id"),
                "question": r.get("question"),
                "signal_score": r.get("signal_score"),
                "max_6h_z": r.get("max_6h_z"),
                "jump_24h_to_1h": r.get("jump_24h_to_1h"),
            }
            for r in rows_cluster[:3]
        ]
        for cluster, rows_cluster in by_cluster.items()
    }

    seeded_summary = summarize_scores(seeded)
    non_seeded_summary = summarize_scores(non_seeded)
    non_seeded_event_summary = summarize_scores(non_seeded_event_eligible)

    effect_vs_event_controls = {
        "mean_delta": seeded_summary["mean"] - non_seeded_event_summary["mean"],
        "share_ge_10_delta": seeded_summary["share_ge_10"] - non_seeded_event_summary["share_ge_10"],
        "share_ge_20_delta": seeded_summary["share_ge_20"] - non_seeded_event_summary["share_ge_20"],
    }

    return {
        "objective": (
            "Assess whether abrupt pre-event price spikes are reliable, repeatable signals by comparing seeded "
            "suspicious events against a broad comparator set of non-seeded markets from the same pull."
        ),
        "research_question": (
            "Do seeded suspicious markets show stronger and more repeatable anomaly signatures than non-seeded "
            "comparators, especially in late windows (T-24h, T-6h, T-1h)?"
        ),
        "dataset": {
            "platform": "Polymarket",
            "window": f"{START_DATE_MIN} to {START_DATE_MAX}",
            "total_markets": len(rows),
            "seeded_markets": len(seeded),
            "non_seeded_markets": len(non_seeded),
            "non_seeded_event_eligible_markets": len(non_seeded_event_eligible),
            "note": "Comparators are non-seeded markets from the same pulled set; this is not yet a global random-sample baseline.",
        },
        "groups": {
            "seeded": seeded_summary,
            "non_seeded": non_seeded_summary,
            "non_seeded_event_eligible": non_seeded_event_summary,
        },
        "effect_vs_event_controls": effect_vs_event_controls,
        "seeded_top": [
            {
                "market_id": r.get("market_id"),
                "question": r.get("question"),
                "signal_score": r.get("signal_score"),
                "cluster": infer_cluster(r),
                "max_6h_z": r.get("max_6h_z"),
                "jump_24h_to_1h": r.get("jump_24h_to_1h"),
            }
            for r in seeded[:8]
        ],
        "non_seeded_top": [
            {
                "market_id": r.get("market_id"),
                "question": r.get("question"),
                "signal_score": r.get("signal_score"),
                "cluster": infer_cluster(r),
            }
            for r in non_seeded[:10]
        ],
        "cluster_top": cluster_top,
    }


def build_analytic_prompt(rows: List[Dict[str, Any]]) -> str:
    payload = build_experiment_packet(rows)

    return f"""You are Nehanda writing an experimental memo.

Write concise markdown with EXACTLY these section headings:
1. Executive Summary (So What)
2. Objective and Hypothesis
3. Dataset and Sampling
4. Experimental Design
5. Results
6. Reliability and Repeatability
7. Decision Use and Next Steps

Hard requirements:
- Answer the research question directly.
- Use concrete numbers from the JSON packet.
- Compare seeded vs non-seeded event-eligible controls.
- Explicitly mention that this comparator is from the same pulled set, not a global random baseline.
- Include concrete market IDs in Results.
- State this score is an anomaly heuristic, not a calibrated event probability.
- Keep language plain and action-oriented.

Data packet (JSON):
{json.dumps(payload, indent=2)}
"""


REQUIRED_SECTION_HEADINGS = [
    "Executive Summary (So What)",
    "Objective and Hypothesis",
    "Dataset and Sampling",
    "Experimental Design",
    "Results",
    "Reliability and Repeatability",
    "Decision Use and Next Steps",
]


def normalize_report_markdown(report: str) -> str:
    lines = report.splitlines()
    out_lines: List[str] = []

    has_title = any(line.strip().startswith("# ") for line in lines)
    if not has_title:
        out_lines.append("# Polymarket Experimental Memo")
        out_lines.append("")

    for line in lines:
        stripped = line.strip()
        if stripped in REQUIRED_SECTION_HEADINGS:
            out_lines.append(f"## {stripped}")
        else:
            out_lines.append(line)

    normalized = "\n".join(out_lines).strip()
    return normalized + "\n"


def extract_sections(report: str) -> Dict[str, str]:
    sections: Dict[str, List[str]] = {}
    current: Optional[str] = None

    for line in report.splitlines():
        heading_match = re.match(r"^\s{0,3}#{1,6}\s+(.+?)\s*$", line)
        if heading_match:
            title = heading_match.group(1).strip()
            if title in REQUIRED_SECTION_HEADINGS:
                current = title
                sections[current] = []
            else:
                current = None
            continue

        if current is not None:
            sections[current].append(line)

    return {k: "\n".join(v).strip() for k, v in sections.items()}


def validate_report_against_hypothesis(report: str, rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    checks_passed: List[str] = []
    checks_failed: List[str] = []

    packet = build_experiment_packet(rows)
    normalized = normalize_report_markdown(report)
    sections = extract_sections(normalized)
    lower = normalized.lower()

    for heading in REQUIRED_SECTION_HEADINGS:
        if heading in sections:
            checks_passed.append(f"section:{heading}")
        else:
            checks_failed.append(f"missing_section:{heading}")

    summary = sections.get("Executive Summary (So What)", "").lower()
    if "so what" in summary or "action" in summary or "operational" in summary:
        checks_passed.append("summary_has_so_what")
    else:
        checks_failed.append("summary_missing_so_what")

    obj = sections.get("Objective and Hypothesis", "").lower()
    if "hypothesis" in obj and "seeded" in obj and "control" in obj:
        checks_passed.append("objective_states_case_control_hypothesis")
    else:
        checks_failed.append("objective_missing_case_control_hypothesis")

    data_scope = sections.get("Dataset and Sampling", "")
    if str(packet["dataset"]["total_markets"]) in data_scope and str(packet["dataset"]["seeded_markets"]) in data_scope:
        checks_passed.append("dataset_mentions_counts")
    else:
        checks_failed.append("dataset_missing_counts")

    if "not yet a global random" in sections.get("Dataset and Sampling", "").lower() or "not a global random" in lower:
        checks_passed.append("dataset_declares_non_random_baseline")
    else:
        checks_failed.append("dataset_missing_non_random_baseline_disclosure")

    results = sections.get("Results", "")
    results_lower = results.lower()
    if "seeded" in results_lower and "control" in results_lower:
        checks_passed.append("results_compares_seeded_and_controls")
    else:
        checks_failed.append("results_missing_seeded_control_comparison")

    if re.search(r"`?\d{5,}`?", results):
        checks_passed.append("results_include_market_ids")
    else:
        checks_failed.append("results_missing_market_ids")

    reliability = sections.get("Reliability and Repeatability", "").lower()
    if "repeat" in reliability and ("threshold" in reliability or ">=" in reliability):
        checks_passed.append("reliability_mentions_repeatability_thresholds")
    else:
        checks_failed.append("reliability_missing_repeatability_thresholds")

    if "anomaly heuristic" in lower and "not a calibrated" in lower:
        checks_passed.append("score_non_probability_disclaimer_present")
    else:
        checks_failed.append("score_non_probability_disclaimer_missing")

    return {
        "passed": len(checks_failed) == 0,
        "passed_checks": checks_passed,
        "failed_checks": checks_failed,
        "normalized_report": normalized,
    }


def build_deterministic_fallback(rows: List[Dict[str, Any]], reason: str) -> str:
    packet = build_experiment_packet(rows)
    seeded = packet["groups"]["seeded"]
    controls = packet["groups"]["non_seeded_event_eligible"]
    effect = packet["effect_vs_event_controls"]

    seeded_top = packet["seeded_top"]
    non_seeded_top = packet["non_seeded_top"]
    cluster_top = packet["cluster_top"]

    if effect["share_ge_10_delta"] > 0 and effect["mean_delta"] > 0:
        verdict = "partial support"
    else:
        verdict = "weak support"

    lines: List[str] = []
    lines.append("# Polymarket Experimental Memo")
    lines.append("")

    lines.append("## Executive Summary (So What)")
    lines.append(
        f"So what: {verdict.capitalize()} for the leak-signal hypothesis in this run. "
        "Strong late-window anomalies are concentrated in a small Iran strike-timing subset, so this is usable as a "
        "targeted alerting screen, not a broad all-market predictor."
    )
    lines.append("")

    lines.append("## Objective and Hypothesis")
    lines.append("- Objective: detect whether abrupt pre-event spikes can be treated as reliable, repeatable warning signals.")
    lines.append(
        "- Hypothesis: seeded suspicious contracts should show stronger anomaly signatures than non-seeded controls, "
        "especially near market close."
    )
    lines.append("")

    lines.append("## Dataset and Sampling")
    lines.append(f"- Platform and window: Polymarket, {packet['dataset']['window']}.")
    lines.append(f"- Total audited markets: {packet['dataset']['total_markets']}.")
    lines.append(f"- Seeded suspicious set: {packet['dataset']['seeded_markets']} markets (Maduro, Khamenei, Iran-strike cases).")
    lines.append(f"- Non-seeded comparators: {packet['dataset']['non_seeded_markets']} markets.")
    lines.append(f"- Event-eligible non-seeded controls (speech/process filtered out): {packet['dataset']['non_seeded_event_eligible_markets']}.")
    lines.append(f"- Sampling note: {packet['dataset']['note']}")
    lines.append(f"- Generation mode: deterministic fallback (`{reason}`).")
    lines.append("")

    lines.append("## Experimental Design")
    lines.append("- Case-control design: seeded suspicious markets vs non-seeded comparators from the same pulled set.")
    lines.append("- Features: max_6h_z, jump_24h_to_1h, range_6h using T-24h/T-6h/T-1h windows.")
    lines.append("- Score: 0.50*max_6h_z + 0.30*(jump_24h_to_1h*100) + 0.20*(range_6h*100).")
    lines.append("- Score interpretation: anomaly heuristic, not a calibrated event probability.")
    lines.append("")

    lines.append("## Results")
    lines.append(
        f"- Seeded vs event-eligible controls mean score: {fmt_num(seeded['mean'])} vs {fmt_num(controls['mean'])} "
        f"(delta {fmt_num(effect['mean_delta'])})."
    )
    lines.append(
        f"- Share with score >=10: seeded {fmt_num(seeded['share_ge_10']*100,2)}% vs controls "
        f"{fmt_num(controls['share_ge_10']*100,2)}% (delta {fmt_num(effect['share_ge_10_delta']*100,2)} pp)."
    )
    lines.append(
        f"- Share with score >=20: seeded {fmt_num(seeded['share_ge_20']*100,2)}% vs controls "
        f"{fmt_num(controls['share_ge_20']*100,2)}% (delta {fmt_num(effect['share_ge_20_delta']*100,2)} pp)."
    )
    lines.append("- Highest seeded IDs:")
    for r in seeded_top[:5]:
        lines.append(
            f"- `{r.get('market_id')}` score={fmt_num(r.get('signal_score'))} [{r.get('cluster')}] :: {r.get('question')}"
        )
    lines.append("- Highest non-seeded comparator IDs:")
    for r in non_seeded_top[:4]:
        lines.append(
            f"- `{r.get('market_id')}` score={fmt_num(r.get('signal_score'))} [{r.get('cluster')}] :: {r.get('question')}"
        )
    lines.append("")

    lines.append("## Reliability and Repeatability")
    lines.append("- Repeatability is uneven across case clusters:")
    for cluster in ["Iran Strikes", "Maduro", "Khamenei"]:
        tops = cluster_top.get(cluster, [])
        if not tops:
            lines.append(f"- {cluster}: no matched markets.")
            continue
        top = tops[0]
        lines.append(
            f"- {cluster}: top `{top.get('market_id')}` score={fmt_num(top.get('signal_score'))}, "
            f"jump24to1={fmt_num(top.get('jump_24h_to_1h'),4)}, z={fmt_num(top.get('max_6h_z'),3)}."
        )
    lines.append(
        "- Threshold pattern (>=10, >=20) supports targeted repeatability in the Iran strike-timing subset, "
        "while Maduro and Khamenei clusters remain weak in this dataset."
    )
    lines.append("")

    lines.append("## Decision Use and Next Steps")
    lines.append("- Use now: treat sudden late-window spikes in targeted strike-timing contracts as an alert trigger for deeper review.")
    lines.append("- Do not use now: broad all-market automation without stronger random-baseline benchmarking.")
    lines.append("1. Run the same scoring pipeline on a true random closed-market sample to establish baseline false-positive rates.")
    lines.append("2. Add first-public-news timestamps and measure lead/lag to test whether spikes systematically precede public disclosure.")
    lines.append("3. Add wallet concentration metrics (top-wallet share, HHI) to test whether spikes are diffuse or concentrated.")

    return "\n".join(lines)


def generate_llm_analytic_report(rows: List[Dict[str, Any]]) -> Tuple[str, str, Dict[str, Any]]:
    token = get_hf_token()
    base_url = HF_NEHANDA_ENDPOINT_URL.rstrip("/")

    prompt = build_analytic_prompt(rows)
    attempt_errors: Dict[str, str] = {}
    endpoint_attempts: List[Dict[str, Any]] = []

    if token:
        headers = {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json",
        }

        mode = "hf_inputs_root"
        url = base_url
        payload = {
            "inputs": prompt,
            "parameters": {
                "max_new_tokens": HF_ANALYTIC_REPORT_MAX_TOKENS,
                "do_sample": True,
                "temperature": 0.7,
                "top_p": 0.95,
            },
        }

        max_attempts = 4
        retryable_statuses = {429, 500, 502, 503, 504}

        for attempt in range(1, max_attempts + 1):
            response_payload, error, status = post_json(url, payload, headers=headers, timeout=180)
            endpoint_attempts.append({
                "attempt": attempt,
                "status": status,
                "error": error,
            })

            if error:
                err_lower = error.lower()
                retryable_error = (
                    status in retryable_statuses
                    or "timed out" in err_lower
                    or "temporarily" in err_lower
                    or "connection reset" in err_lower
                    or "service unavailable" in err_lower
                )
                if retryable_error and attempt < max_attempts:
                    time.sleep(min(2 ** (attempt - 1), 8))
                    continue

                attempt_errors[mode] = error
                break

            text = extract_generated_text(response_payload)
            if not text or not text.strip():
                attempt_errors[mode] = f"empty or unparseable response: {str(response_payload)[:200]}"
                break

            validation = validate_report_against_hypothesis(text.strip(), rows)
            if validation["passed"]:
                return (
                    validation["normalized_report"],
                    mode,
                    {
                        "status": status,
                        "attempt_errors": attempt_errors,
                        "endpoint_attempts": endpoint_attempts,
                        "endpoint_contract": "POST / with {inputs, parameters}",
                        "passed_checks": validation["passed_checks"],
                        "failed_checks": validation["failed_checks"],
                        "used_fallback": False,
                    },
                )

            attempt_errors[mode] = (
                "validation failed: " + "; ".join(validation["failed_checks"][:8])
            )
            break

    if not token:
        fallback_reason = "missing HF token"
    elif any("validation failed" in msg for msg in attempt_errors.values()):
        fallback_reason = "model output failed report validation on root endpoint"
    else:
        fallback_reason = "endpoint unavailable/unparseable after root-route retries"
    fallback = build_deterministic_fallback(rows, fallback_reason)
    fallback_validation = validate_report_against_hypothesis(fallback, rows)

    return (
        fallback_validation["normalized_report"],
        "deterministic_fallback",
        {
            "error": None if token else "missing HF token",
            "attempt_errors": attempt_errors,
            "endpoint_attempts": endpoint_attempts,
            "endpoint_contract": "POST / with {inputs, parameters}",
            "fallback_reason": fallback_reason,
            "passed_checks": fallback_validation["passed_checks"],
            "failed_checks": fallback_validation["failed_checks"],
            "used_fallback": True,
        },
    )


if not audit_rows:
    raise RuntimeError("audit_rows is empty. Run prior analysis cells first.")

analytic_report, analytic_source, analytic_meta = generate_llm_analytic_report(audit_rows)
analytic_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
analytic_md_path = os.path.join(REPORT_DIR, f"polymarket_analytic_report_{analytic_timestamp}.md")
analytic_latest_path = os.path.join(REPORT_DIR, "polymarket_analytic_report_latest.md")
analytic_meta_path = os.path.join(REPORT_DIR, f"polymarket_analytic_report_meta_{analytic_timestamp}.json")
analytic_meta_latest_path = os.path.join(REPORT_DIR, "polymarket_analytic_report_meta_latest.json")

with open(analytic_md_path, "w", encoding="utf-8") as f:
    f.write(analytic_report)
with open(analytic_latest_path, "w", encoding="utf-8") as f:
    f.write(analytic_report)

with open(analytic_meta_path, "w", encoding="utf-8") as f:
    json.dump({"source": analytic_source, "meta": analytic_meta}, f, indent=2)
with open(analytic_meta_latest_path, "w", encoding="utf-8") as f:
    json.dump({"source": analytic_source, "meta": analytic_meta}, f, indent=2)

print(f"Analytic source: {analytic_source}")
print(f"Wrote report: {analytic_md_path}")
print(f"Wrote report latest: {analytic_latest_path}")
print(f"Wrote metadata: {analytic_meta_path}")
print(f"Wrote metadata latest: {analytic_meta_latest_path}")







Analytic source: deterministic_fallback
Wrote report: reports/polymarket_analytic_report_20260307T010341Z.md
Wrote report latest: reports/polymarket_analytic_report_latest.md
Wrote metadata: reports/polymarket_analytic_report_meta_20260307T010341Z.json
Wrote metadata latest: reports/polymarket_analytic_report_meta_latest.json


## Optional: Trade Concentration (requires your authenticated trade headers)

If your existing credentials can query `GET /trades`, define headers and run a concentration add-on.

Example placeholder:
```python
CLOB_TRADE_HEADERS = {
    # Fill with your working auth headers for clob.polymarket.com/trades
}
```

Then compute wallet concentration metrics for `T-6h` (e.g., top-wallet share, HHI) and join onto `audit_rows`.
Keep this as a separate enrichment step so the core notebook works with public endpoints only.
